In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Load data
try:
    df = pd.read_parquet("data/analysis_sample.parquet")
except Exception:
    df = pd.read_csv(r"C:\Users\ugochukwu.emenike\PycharmProjects\PythonProject\powerbi\analysis_sample.csv")

# 2. Prepare target & remove leakage
df = df.dropna(subset=['SYSTOLIC_BP', 'DIASTOLIC_BP']).copy()
df['HIGH_BP_FLAG'] = np.where((df['SYSTOLIC_BP'] >= 140) | (df['DIASTOLIC_BP'] >= 90), 1, 0)

leakage_cols = ['SYSTOLIC_BP', 'DIASTOLIC_BP', 'DIAGNOSED_HYPERTENSION', 'BP_CATEGORY', 'SAMPLE_ID', 'HIGH_BP_FLAG']
X = df.drop(columns=[col for col in leakage_cols if col in df.columns])
y = df['HIGH_BP_FLAG']

# 3. Configure preprocessor
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features),
        ('cat', cat_pipeline, categorical_features)
    ]
)

# 4. Set up models inside complete pipelines
log_reg_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))
])

# 5. Evaluate with 5-Fold Stratified Cross-Validation (cv=5 preserves class balance in every fold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Running 5-Fold CV for Logistic Regression...")
log_reg_scores = cross_val_score(log_reg_pipe, X, y, cv=cv, scoring='roc_auc')

print("Running 5-Fold CV for Random Forest...")
rf_scores = cross_val_score(rf_pipe, X, y, cv=cv, scoring='roc_auc')

# 6. Build the summary table
cv_summary = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Fold Scores (ROC-AUC)": [round(s, 4) for s in log_reg_scores],
        "Mean ROC-AUC": round(np.mean(log_reg_scores), 4),
        "Std Dev (±)": round(np.std(log_reg_scores), 4)
    },
    {
        "Model": "Random Forest",
        "Fold Scores (ROC-AUC)": [round(s, 4) for s in rf_scores],
        "Mean ROC-AUC": round(np.mean(rf_scores), 4),
        "Std Dev (±)": round(np.std(rf_scores), 4)
    }
])

print("\n=== 5-Fold Cross-Validation Performance (ROC-AUC) ===")
display(cv_summary)

Running 5-Fold CV for Logistic Regression...
Running 5-Fold CV for Random Forest...

=== 5-Fold Cross-Validation Performance (ROC-AUC) ===


,Model,Fold Scores (ROC-AUC),Mean ROC-AUC,Std Dev (±)
0,Logistic Regression,"[0.6093, 0.6126, 0.608, 0.6121, 0.6086]",0.6101,0.0019
1,Random Forest,"[0.6013, 0.5962, 0.5982, 0.6022, 0.6053]",0.6006,0.0032


### Cross-Validation Stability Analysis for Ngozi

A high standard deviation across folds indicates that the model is unstable and sensitive to the specific subset of data it trains on, which suggests overfitting or high variance. In our clinical triage context, this would mean the model might perform well in some local testing cohorts but fail unpredictably when deployed across different clinics or geographic zones.